# NLP Sentiment Analysis: DistilBERT Fine-Tuning with Argilla

**Project:** Banking77 Intent Classification using Transfer Learning  
**Model:** `distilbert-base-uncased` (fine-tuned)  
**Dataset:** [Banking77](https://huggingface.co/datasets/PolyAI/banking77) — 13,083 customer service queries across 77 intent categories  
**Annotation Tool:** [Argilla](https://argilla.io/) — Data-centric AI annotation platform  

---

## Pipeline Overview

```
+---------------------+       +---------------------+       +---------------------+
|  1. Pretrained       |       |  2. Argilla          |       |  3. Fine-Tune        |
|     DistilBERT       | ----> |     Annotation       | ----> |     with Trainer     |
|     Inference        |       |     & Labeling       |       |     API              |
+---------------------+       +---------------------+       +---------------------+
         ^                                                            |
         |                                                            |
         +------------------------------------------------------------+
                          Active Learning Loop
```

### What this notebook covers:
1. **Setup & Imports** — All required libraries
2. **Configuration** — Hyperparameters and label mappings
3. **Data Loading & EDA** — Banking77 exploration with visualizations
4. **Argilla Integration** — Annotation workflow and quality metrics
5. **Text Preprocessing** — Tokenization with DistilBERT tokenizer
6. **Model Setup** — DistilBERT for sequence classification
7. **Training** — HuggingFace Trainer with loss tracking
8. **Evaluation** — Classification report, confusion matrix, ROC curves, error analysis
9. **Inference Pipeline** — Prediction pipeline with confidence visualization

---
## 1. Setup & Imports

We use the HuggingFace ecosystem (`transformers`, `datasets`, `accelerate`) together with `argilla` for annotation, `torch` as the deep learning backend, and standard data-science libraries for analysis and visualization.

> **Note:** Run `pip install -r requirements.txt` before executing this notebook. GPU acceleration (CUDA) is recommended for training.

In [ ]:
# ── Core deep learning & NLP ──────────────────────────────────────────────────
import torch
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    pipeline,
    EarlyStoppingCallback,
)
from datasets import load_dataset, DatasetDict, Dataset
import accelerate

# ── Argilla annotation platform ───────────────────────────────────────────────
import argilla as rg

# ── Machine learning utilities ────────────────────────────────────────────────
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    roc_curve,
    auc,
    accuracy_score,
)
from sklearn.preprocessing import label_binarize

# ── Data manipulation ─────────────────────────────────────────────────────────
import pandas as pd
import numpy as np

# ── Visualization ─────────────────────────────────────────────────────────────
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import seaborn as sns
from collections import Counter

# ── Word cloud ────────────────────────────────────────────────────────────────
try:
    from wordcloud import WordCloud
    WORDCLOUD_AVAILABLE = True
except ImportError:
    WORDCLOUD_AVAILABLE = False
    print("[INFO] wordcloud not installed. Run: pip install wordcloud")

# ── Standard library ──────────────────────────────────────────────────────────
import os
import re
import json
import warnings
from pathlib import Path

warnings.filterwarnings("ignore")

# ── Reproducibility ───────────────────────────────────────────────────────────
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# ── Matplotlib style ──────────────────────────────────────────────────────────
plt.style.use("seaborn-v0_8-whitegrid")
sns.set_palette("husl")

# ── Device ────────────────────────────────────────────────────────────────────
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device : {DEVICE}")
print(f"PyTorch : {torch.__version__}")
print(f"Transformers: {__import__('transformers').__version__}")
print(f"Accelerate  : {accelerate.__version__}")

---
## 2. Configuration

Centralising all hyperparameters and paths into a single `CFG` object makes experiments reproducible and easy to modify.

**Banking77** has 77 intents — we group them into three high-level **sentiment buckets** for demonstration:
- `0 → negative` (complaints, disputes, failed transactions)
- `1 → neutral` (informational queries, balance checks)
- `2 → positive` (successful actions, card activations)

In [ ]:
class CFG:
    # ── Model ─────────────────────────────────────────────────────────────────
    MODEL_NAME        = "distilbert-base-uncased"
    MAX_LENGTH        = 128          # tokens (Banking77 queries are short)
    NUM_LABELS        = 3            # negative / neutral / positive

    # ── Training ──────────────────────────────────────────────────────────────
    BATCH_SIZE        = 32
    EVAL_BATCH_SIZE   = 64
    EPOCHS            = 5
    LEARNING_RATE     = 2e-5
    WEIGHT_DECAY      = 0.01
    WARMUP_RATIO      = 0.1          # 10 % of total steps for LR warm-up
    GRADIENT_CLIP     = 1.0
    PATIENCE          = 2            # early-stopping patience

    # ── Label mapping ─────────────────────────────────────────────────────────
    LABEL2ID = {"negative": 0, "neutral": 1, "positive": 2}
    ID2LABEL = {0: "negative", 1: "neutral", 2: "positive"}

    # Banking77 intent → sentiment bucket
    # (selected representative intents for each bucket)
    NEGATIVE_INTENTS = [
        "cancel_transfer", "card_not_working", "declined_card_payment",
        "declined_cash_withdrawal", "declined_transfer", "transaction_charged_twice",
        "fraud_dispute", "compromised_card", "lost_or_stolen_card",
        "lost_or_stolen_phone", "wrong_amount_of_cash_received",
        "wrong_exchange_rate_for_cash_withdrawal",
    ]
    POSITIVE_INTENTS = [
        "activate_my_card", "card_acceptance", "card_arrival",
        "card_delivery_estimate", "card_linking", "card_payment_fee_charged",
        "card_swallowed", "get_physical_card", "request_refund",
        "supported_cards_and_currencies",
    ]
    # All remaining intents are neutral

    # ── Paths ─────────────────────────────────────────────────────────────────
    OUTPUT_DIR        = "../models/distilbert-banking77"
    LOGS_DIR          = "../models/logs"
    DATA_DIR          = "../data"

    # ── Argilla ───────────────────────────────────────────────────────────────
    ARGILLA_API_URL   = "http://localhost:6900"
    ARGILLA_API_KEY   = "argilla.apikey"   # default dev key
    ARGILLA_WORKSPACE = "banking-sentiment"
    ARGILLA_DATASET   = "banking77-annotation"
    ANNOTATION_SAMPLE = 500              # records to push for human review


# Create output directories
for d in [CFG.OUTPUT_DIR, CFG.LOGS_DIR, CFG.DATA_DIR]:
    Path(d).mkdir(parents=True, exist_ok=True)

print("Configuration loaded.")
print(f"  Model        : {CFG.MODEL_NAME}")
print(f"  Max length   : {CFG.MAX_LENGTH} tokens")
print(f"  Batch size   : {CFG.BATCH_SIZE}")
print(f"  Epochs       : {CFG.EPOCHS}")
print(f"  Learning rate: {CFG.LEARNING_RATE}")
print(f"  Labels       : {CFG.ID2LABEL}")

---
## 3. Data Loading & Exploratory Data Analysis

We load **Banking77** from the HuggingFace Hub and map the original 77 intent labels to our three sentiment buckets. EDA covers:
- Class balance
- Text length distribution
- Word clouds per sentiment class
- Sample text inspection

In [ ]:
# ── 3.1  Load Banking77 ───────────────────────────────────────────────────────
print("Loading Banking77 dataset...")
raw_dataset = load_dataset("PolyAI/banking77")

# The dataset has a ClassLabel feature with 77 classes
label_names = raw_dataset["train"].features["label"].names
print(f"Train size : {len(raw_dataset['train']):,}")
print(f"Test size  : {len(raw_dataset['test']):,}")
print(f"Num intents: {len(label_names)}")
print(f"\nFirst 5 intents: {label_names[:5]}")

In [ ]:
# ── 3.2  Map 77-class intents → 3-class sentiment ─────────────────────────────
def intent_to_sentiment(intent_id: int) -> int:
    """Map Banking77 intent index to sentiment label (0/1/2)."""
    intent_name = label_names[intent_id]
    if intent_name in CFG.NEGATIVE_INTENTS:
        return CFG.LABEL2ID["negative"]
    elif intent_name in CFG.POSITIVE_INTENTS:
        return CFG.LABEL2ID["positive"]
    else:
        return CFG.LABEL2ID["neutral"]


def add_sentiment_label(example):
    example["sentiment"] = intent_to_sentiment(example["label"])
    example["intent_name"] = label_names[example["label"]]
    return example


mapped = raw_dataset.map(add_sentiment_label, desc="Mapping sentiments")

# Convert to pandas for EDA
train_df = mapped["train"].to_pandas()
test_df  = mapped["test"].to_pandas()

train_df["sentiment_name"] = train_df["sentiment"].map(CFG.ID2LABEL)
test_df["sentiment_name"]  = test_df["sentiment"].map(CFG.ID2LABEL)

print("Sentiment distribution (train):")
print(train_df["sentiment_name"].value_counts())

In [ ]:
# ── 3.3  Class balance ────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Train split
counts_train = train_df["sentiment_name"].value_counts()
colors = sns.color_palette("husl", 3)
axes[0].bar(counts_train.index, counts_train.values, color=colors, edgecolor="white", linewidth=1.5)
axes[0].set_title("Class Balance — Train Split", fontsize=14, fontweight="bold")
axes[0].set_xlabel("Sentiment")
axes[0].set_ylabel("Count")
for i, (label, count) in enumerate(counts_train.items()):
    axes[0].text(i, count + 20, str(count), ha="center", fontweight="bold")

# Test split
counts_test = test_df["sentiment_name"].value_counts()
axes[1].bar(counts_test.index, counts_test.values, color=colors, edgecolor="white", linewidth=1.5)
axes[1].set_title("Class Balance — Test Split", fontsize=14, fontweight="bold")
axes[1].set_xlabel("Sentiment")
axes[1].set_ylabel("Count")
for i, (label, count) in enumerate(counts_test.items()):
    axes[1].text(i, count + 5, str(count), ha="center", fontweight="bold")

plt.suptitle("Banking77 → 3-Class Sentiment Distribution", fontsize=15, fontweight="bold", y=1.02)
plt.tight_layout()
plt.savefig("../data/class_balance.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# ── 3.4  Text length distribution ─────────────────────────────────────────────
train_df["text_len"]      = train_df["text"].str.len()
train_df["word_count"]    = train_df["text"].str.split().str.len()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for sentiment, color in zip(["negative", "neutral", "positive"], colors):
    subset = train_df[train_df["sentiment_name"] == sentiment]
    axes[0].hist(subset["text_len"], bins=30, alpha=0.6, label=sentiment, color=color)
    axes[1].hist(subset["word_count"], bins=20, alpha=0.6, label=sentiment, color=color)

axes[0].set_title("Character Length Distribution", fontsize=13, fontweight="bold")
axes[0].set_xlabel("Characters")
axes[0].set_ylabel("Frequency")
axes[0].legend()
axes[0].axvline(train_df["text_len"].mean(), color="black", linestyle="--", label=f"mean={train_df['text_len'].mean():.0f}")

axes[1].set_title("Word Count Distribution", fontsize=13, fontweight="bold")
axes[1].set_xlabel("Words")
axes[1].set_ylabel("Frequency")
axes[1].legend()
axes[1].axvline(train_df["word_count"].mean(), color="black", linestyle="--", label=f"mean={train_df['word_count'].mean():.1f}")

plt.suptitle("Text Length Analysis by Sentiment", fontsize=15, fontweight="bold", y=1.02)
plt.tight_layout()
plt.savefig("../data/text_lengths.png", dpi=150, bbox_inches="tight")
plt.show()

print("Text length statistics:")
print(train_df.groupby("sentiment_name")[["text_len", "word_count"]].describe().round(1))

In [ ]:
# ── 3.5  Word clouds per sentiment class ──────────────────────────────────────
if WORDCLOUD_AVAILABLE:
    STOPWORDS = {
        "i", "my", "the", "a", "an", "to", "of", "is", "it", "in",
        "that", "was", "for", "on", "are", "with", "as", "at", "be",
        "this", "have", "from", "or", "one", "had", "by", "but", "not",
        "what", "all", "were", "we", "when", "your", "can", "said", "do",
        "has", "how", "he", "if", "no", "up", "so", "me",
    }

    sentiment_colors = {"negative": "Reds", "neutral": "Blues", "positive": "Greens"}

    fig, axes = plt.subplots(1, 3, figsize=(18, 6))
    for ax, sentiment in zip(axes, ["negative", "neutral", "positive"]):
        subset_text = " ".join(train_df[train_df["sentiment_name"] == sentiment]["text"].tolist())
        wc = WordCloud(
            width=600, height=400,
            background_color="white",
            colormap=sentiment_colors[sentiment],
            stopwords=STOPWORDS,
            max_words=80,
            random_state=SEED,
        ).generate(subset_text)
        ax.imshow(wc, interpolation="bilinear")
        ax.axis("off")
        ax.set_title(f"Word Cloud — {sentiment.capitalize()}", fontsize=14, fontweight="bold", pad=10)

    plt.suptitle("Most Frequent Words per Sentiment Class", fontsize=15, fontweight="bold", y=1.02)
    plt.tight_layout()
    plt.savefig("../data/wordclouds.png", dpi=150, bbox_inches="tight")
    plt.show()
else:
    print("[SKIP] Install wordcloud to generate word cloud visualizations.")

In [ ]:
# ── 3.6  Sample texts per class ───────────────────────────────────────────────
print("=" * 70)
for sentiment in ["negative", "neutral", "positive"]:
    print(f"\n{'='*70}")
    print(f"  {sentiment.upper()} EXAMPLES")
    print(f"{'='*70}")
    samples = train_df[train_df["sentiment_name"] == sentiment][["text", "intent_name"]].sample(3, random_state=SEED)
    for _, row in samples.iterrows():
        print(f"  [{row['intent_name']}]")
        print(f"  {row['text']}")
        print()

In [ ]:
# ── 3.7  Top intents per sentiment heatmap ────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(18, 7))

for ax, sentiment in zip(axes, ["negative", "neutral", "positive"]):
    subset = train_df[train_df["sentiment_name"] == sentiment]
    top_intents = subset["intent_name"].value_counts().head(10)
    ax.barh(top_intents.index[::-1], top_intents.values[::-1],
            color=sns.color_palette("husl", len(top_intents)))
    ax.set_title(f"Top 10 Intents — {sentiment.capitalize()}", fontsize=12, fontweight="bold")
    ax.set_xlabel("Count")
    ax.tick_params(axis="y", labelsize=9)

plt.suptitle("Intent Distribution within Each Sentiment Bucket", fontsize=14, fontweight="bold", y=1.02)
plt.tight_layout()
plt.savefig("../data/intent_distributions.png", dpi=150, bbox_inches="tight")
plt.show()

---
## 4. Argilla Integration

[Argilla](https://docs.argilla.io/) is an open-source data annotation and labeling platform built for NLP. It enables human-in-the-loop quality control and supports the **active learning** loop at the core of this project.

### Active learning workflow:
1. **Predict** on unlabelled/uncertain samples with the current model
2. **Log to Argilla** for human review — annotators verify or correct predictions
3. **Query annotated data** back into the training pipeline
4. **Retrain** with the cleaned, human-verified labels
5. Repeat

> **Prerequisites:** Start the Argilla server with `argilla server start` before running this section.

In [ ]:
# ── 4.1  Connect to Argilla ───────────────────────────────────────────────────
try:
    rg.init(
        api_url=CFG.ARGILLA_API_URL,
        api_key=CFG.ARGILLA_API_KEY,
        workspace=CFG.ARGILLA_WORKSPACE,
    )
    print(f"Connected to Argilla at {CFG.ARGILLA_API_URL}")
    ARGILLA_CONNECTED = True
except Exception as e:
    print(f"[WARNING] Could not connect to Argilla: {e}")
    print("Continuing in offline mode — annotation steps will be simulated.")
    ARGILLA_CONNECTED = False

In [ ]:
# ── 4.2  Define Argilla dataset schema ────────────────────────────────────────
# We use TextClassificationRecord: each record has the raw text,
# model predictions (as prediction probabilities), and optionally
# a human annotation for correction.

def create_argilla_records(
    df: pd.DataFrame,
    model_predictions: list[dict] | None = None,
    n_samples: int = CFG.ANNOTATION_SAMPLE,
) -> list:
    """Build Argilla TextClassificationRecord objects from a DataFrame."""
    sample = df.sample(n=min(n_samples, len(df)), random_state=SEED).reset_index(drop=True)
    records = []

    for idx, row in sample.iterrows():
        # Build prediction field (model confidence scores)
        if model_predictions and idx < len(model_predictions):
            pred = model_predictions[idx]
            prediction = [(label, score) for label, score in pred.items()]
            pred_agent = CFG.MODEL_NAME
        else:
            prediction = None
            pred_agent = None

        record = rg.TextClassificationRecord(
            text=row["text"],
            prediction=prediction,
            prediction_agent=pred_agent,
            annotation=row["sentiment_name"],   # auto-annotation from intent mapping
            annotation_agent="intent-mapping",
            metadata={
                "intent": row["intent_name"],
                "split": "train" if idx < len(train_df) else "test",
            },
            id=str(idx),
        )
        records.append(record)

    return records


print(f"Argilla record builder ready.")
print(f"Will create up to {CFG.ANNOTATION_SAMPLE} records for human review.")

In [ ]:
# ── 4.3  Log samples to Argilla ───────────────────────────────────────────────
if ARGILLA_CONNECTED:
    records = create_argilla_records(train_df, n_samples=CFG.ANNOTATION_SAMPLE)

    rg.log(
        records=records,
        name=CFG.ARGILLA_DATASET,
        workspace=CFG.ARGILLA_WORKSPACE,
    )
    print(f"Logged {len(records)} records to Argilla dataset '{CFG.ARGILLA_DATASET}'.")
    print(f"Review at: {CFG.ARGILLA_API_URL}/datasets/{CFG.ARGILLA_WORKSPACE}/{CFG.ARGILLA_DATASET}")
else:
    print("[OFFLINE] Skipping Argilla log. Records would be:")
    # Simulate what records look like
    sim_records = train_df.sample(5, random_state=SEED)[["text", "sentiment_name", "intent_name"]]
    print(sim_records.to_string(index=False))

In [ ]:
# ── 4.4  Query annotated data back from Argilla ───────────────────────────────
if ARGILLA_CONNECTED:
    # Load all ANNOTATED records (status="Validated" means human-reviewed)
    annotated_ds = rg.load(
        name=CFG.ARGILLA_DATASET,
        workspace=CFG.ARGILLA_WORKSPACE,
        query="status:Validated",
    )
    annotated_df = annotated_ds.to_pandas()
    print(f"Retrieved {len(annotated_df)} validated records from Argilla.")

    if len(annotated_df) > 0:
        # Compare auto-annotation vs human annotation to measure agreement
        annotated_df["auto_label"]   = annotated_df["annotation"]  # our mapping
        annotated_df["human_label"]  = annotated_df["annotation"]  # human reviewer

        # Agreement rate
        agreement = (annotated_df["auto_label"] == annotated_df["human_label"]).mean()
        print(f"\nAnnotation agreement (auto vs. human): {agreement:.1%}")
else:
    print("[OFFLINE] Simulating annotation quality metrics...")

    # Simulate realistic annotation agreement
    np.random.seed(SEED)
    n_sim = 200
    auto_labels   = np.random.choice(["negative", "neutral", "positive"], size=n_sim, p=[0.2, 0.6, 0.2])
    # Human annotators agree ~88% of the time on clear cases
    noise_mask    = np.random.random(n_sim) < 0.12
    human_labels  = auto_labels.copy()
    for i in np.where(noise_mask)[0]:
        choices = [l for l in ["negative", "neutral", "positive"] if l != auto_labels[i]]
        human_labels[i] = np.random.choice(choices)

    sim_annotation_df = pd.DataFrame({"auto_label": auto_labels, "human_label": human_labels})
    agreement = (sim_annotation_df["auto_label"] == sim_annotation_df["human_label"]).mean()
    print(f"Simulated annotation agreement: {agreement:.1%} ({n_sim} records)")

In [ ]:
# ── 4.5  Annotation quality metrics ───────────────────────────────────────────
# Visualise inter-annotator agreement and correction patterns

if not ARGILLA_CONNECTED:
    # Use simulated data
    auto_labels_plot  = sim_annotation_df["auto_label"]
    human_labels_plot = sim_annotation_df["human_label"]
    n_reviewed = len(sim_annotation_df)
else:
    auto_labels_plot  = annotated_df["auto_label"]
    human_labels_plot = annotated_df["human_label"]
    n_reviewed = len(annotated_df)

# Agreement confusion matrix
label_order = ["negative", "neutral", "positive"]
cm_annot = confusion_matrix(auto_labels_plot, human_labels_plot, labels=label_order)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Confusion matrix heatmap
sns.heatmap(
    cm_annot, annot=True, fmt="d", cmap="Blues",
    xticklabels=label_order, yticklabels=label_order,
    ax=axes[0], cbar=True,
)
axes[0].set_title("Annotation Agreement\n(Auto-label vs. Human-label)", fontweight="bold")
axes[0].set_xlabel("Human Label")
axes[0].set_ylabel("Auto Label")

# Correction rate per class
correction_rates = {}
for lbl in label_order:
    mask = auto_labels_plot == lbl
    if mask.sum() > 0:
        correction_rates[lbl] = 1 - (auto_labels_plot[mask] == human_labels_plot[mask]).mean()
    else:
        correction_rates[lbl] = 0

axes[1].bar(correction_rates.keys(), correction_rates.values(),
            color=colors, edgecolor="white", linewidth=1.5)
axes[1].set_title("Human Correction Rate per Class\n(Lower = more reliable auto-labelling)", fontweight="bold")
axes[1].set_xlabel("Sentiment")
axes[1].set_ylabel("Correction Rate")
axes[1].set_ylim(0, 0.5)
for i, (lbl, rate) in enumerate(correction_rates.items()):
    axes[1].text(i, rate + 0.005, f"{rate:.1%}", ha="center", fontweight="bold")

plt.suptitle(f"Argilla Annotation Quality ({n_reviewed} reviewed records)",
             fontsize=14, fontweight="bold", y=1.02)
plt.tight_layout()
plt.savefig("../data/annotation_quality.png", dpi=150, bbox_inches="tight")
plt.show()

print(f"\nOverall annotation agreement : {agreement:.1%}")
print(f"Correction rates by class    : {correction_rates}")

---
## 5. Text Preprocessing

DistilBERT requires inputs to be **tokenized** into subword units using its WordPiece vocabulary. Key steps:
- **Tokenization** with `distilbert-base-uncased` tokenizer
- **Truncation** to `MAX_LENGTH` tokens (Banking77 queries are short — rarely exceeds 50 tokens)
- **Padding** to uniform length within each batch
- Converting to HuggingFace `Dataset` objects for efficient training

In [ ]:
# ── 5.1  Load DistilBERT tokenizer ────────────────────────────────────────────
print(f"Loading tokenizer: {CFG.MODEL_NAME}")
tokenizer = AutoTokenizer.from_pretrained(CFG.MODEL_NAME)

# Inspect a sample encoding
sample_text = "I can't log into my banking app and my card has been blocked."
encoding = tokenizer(sample_text, truncation=True, max_length=CFG.MAX_LENGTH, padding="max_length")

print(f"\nSample text : {sample_text}")
print(f"Tokens      : {tokenizer.convert_ids_to_tokens(encoding['input_ids'][:15])} ...")
print(f"Input IDs   : {encoding['input_ids'][:15]} ...")
print(f"Attention   : {encoding['attention_mask'][:15]} ...")
print(f"Token count : {sum(encoding['attention_mask'])} / {CFG.MAX_LENGTH}")

In [ ]:
# ── 5.2  Token length distribution ────────────────────────────────────────────
# Check that MAX_LENGTH = 128 covers the vast majority of examples
token_lengths = [
    len(tokenizer.encode(text, truncation=False))
    for text in train_df["text"].tolist()
]

fig, ax = plt.subplots(figsize=(10, 4))
ax.hist(token_lengths, bins=40, color="steelblue", edgecolor="white", linewidth=0.8)
ax.axvline(CFG.MAX_LENGTH, color="red", linestyle="--", linewidth=2, label=f"MAX_LENGTH = {CFG.MAX_LENGTH}")
ax.axvline(np.percentile(token_lengths, 95), color="orange", linestyle="--",
           linewidth=2, label=f"95th percentile = {np.percentile(token_lengths, 95):.0f}")
ax.set_title("Token Length Distribution (DistilBERT tokenizer)", fontsize=13, fontweight="bold")
ax.set_xlabel("Number of tokens")
ax.set_ylabel("Frequency")
ax.legend()

truncated = sum(1 for l in token_lengths if l > CFG.MAX_LENGTH)
ax.text(0.97, 0.95, f"Truncated: {truncated}/{len(token_lengths)} ({truncated/len(token_lengths):.1%})",
        transform=ax.transAxes, ha="right", va="top",
        bbox=dict(boxstyle="round", facecolor="wheat", alpha=0.8))

plt.tight_layout()
plt.savefig("../data/token_lengths.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"Max token length: {max(token_lengths)}  |  Mean: {np.mean(token_lengths):.1f}  |  p95: {np.percentile(token_lengths, 95):.0f}")

In [ ]:
# ── 5.3  Tokenize the full dataset ────────────────────────────────────────────
def tokenize_fn(batch):
    """Tokenize a batch of texts."""
    return tokenizer(
        batch["text"],
        truncation=True,
        max_length=CFG.MAX_LENGTH,
        padding="max_length",  # pad to max_length for consistent batching
    )


# Build HuggingFace Dataset objects from pandas
train_hf = Dataset.from_pandas(train_df[["text", "sentiment"]].rename(columns={"sentiment": "labels"}))
test_hf  = Dataset.from_pandas(test_df[["text", "sentiment"]].rename(columns={"sentiment": "labels"}))

# Apply tokenization (batched for speed)
train_tokenized = train_hf.map(tokenize_fn, batched=True, desc="Tokenizing train")
test_tokenized  = test_hf.map(tokenize_fn, batched=True, desc="Tokenizing test")

# Set format to PyTorch tensors
cols = ["input_ids", "attention_mask", "labels"]
train_tokenized.set_format(type="torch", columns=cols)
test_tokenized.set_format(type="torch", columns=cols)

print("Tokenization complete.")
print(f"  Train features: {train_tokenized.features}")
print(f"  Train size    : {len(train_tokenized):,}")
print(f"  Test size     : {len(test_tokenized):,}")

# Quick sanity check
sample_batch = train_tokenized[0]
print(f"\nSingle example:")
print(f"  input_ids shape     : {sample_batch['input_ids'].shape}")
print(f"  attention_mask shape: {sample_batch['attention_mask'].shape}")
print(f"  label               : {sample_batch['labels'].item()} → {CFG.ID2LABEL[sample_batch['labels'].item()]}")

---
## 6. Model Setup

We load `distilbert-base-uncased` from the HuggingFace Hub and replace the default classification head with one sized for our 3-class problem.

**DistilBERT architecture:**
- 6 transformer layers (vs. BERT's 12)
- 768 hidden dimensions
- 66M parameters (40% fewer than BERT-base)
- Pre-trained via knowledge distillation from BERT-base

The `TrainingArguments` configure all aspects of the training loop: learning rate schedule, gradient clipping, evaluation strategy, and checkpoint saving.

In [ ]:
# ── 6.1  Load model ───────────────────────────────────────────────────────────
print(f"Loading {CFG.MODEL_NAME} for sequence classification...")

model = AutoModelForSequenceClassification.from_pretrained(
    CFG.MODEL_NAME,
    num_labels=CFG.NUM_LABELS,
    id2label=CFG.ID2LABEL,
    label2id=CFG.LABEL2ID,
)
model = model.to(DEVICE)

# Count parameters
total_params    = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"\nModel loaded on {DEVICE}")
print(f"  Total parameters    : {total_params:,}")
print(f"  Trainable parameters: {trainable_params:,}")
print(f"  Classification head : distilbert-base → Linear({model.config.dim}, {CFG.NUM_LABELS})")

In [ ]:
# ── 6.2  Training arguments ───────────────────────────────────────────────────
total_steps   = (len(train_tokenized) // CFG.BATCH_SIZE) * CFG.EPOCHS
warmup_steps  = int(total_steps * CFG.WARMUP_RATIO)

print(f"Total training steps : {total_steps:,}")
print(f"Warmup steps         : {warmup_steps:,}  ({CFG.WARMUP_RATIO:.0%} of total)")

training_args = TrainingArguments(
    output_dir=CFG.OUTPUT_DIR,
    logging_dir=CFG.LOGS_DIR,

    # ── Batch & epochs ────────────────────────────────────────────────────────
    num_train_epochs=CFG.EPOCHS,
    per_device_train_batch_size=CFG.BATCH_SIZE,
    per_device_eval_batch_size=CFG.EVAL_BATCH_SIZE,

    # ── Optimiser ─────────────────────────────────────────────────────────────
    learning_rate=CFG.LEARNING_RATE,
    weight_decay=CFG.WEIGHT_DECAY,
    max_grad_norm=CFG.GRADIENT_CLIP,
    warmup_steps=warmup_steps,
    lr_scheduler_type="linear",       # linear decay after warmup

    # ── Evaluation & checkpoints ──────────────────────────────────────────────
    evaluation_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    greater_is_better=True,

    # ── Logging ───────────────────────────────────────────────────────────────
    logging_steps=50,
    report_to="none",                 # disable W&B / MLflow for this notebook

    # ── Reproducibility ───────────────────────────────────────────────────────
    seed=SEED,
    data_seed=SEED,

    # ── Efficiency ────────────────────────────────────────────────────────────
    fp16=torch.cuda.is_available(),   # mixed precision on GPU
    dataloader_num_workers=0,
)

print("\nTrainingArguments configured.")
print(f"  Eval strategy : {training_args.evaluation_strategy}")
print(f"  Best metric   : {training_args.metric_for_best_model}")
print(f"  FP16          : {training_args.fp16}")

In [ ]:
# ── 6.3  Metrics function ─────────────────────────────────────────────────────
from sklearn.metrics import f1_score

def compute_metrics(eval_pred):
    """
    Compute accuracy and macro-F1 from Trainer evaluation outputs.
    Called after each evaluation epoch.
    """
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)

    acc      = accuracy_score(labels, preds)
    f1_macro = f1_score(labels, preds, average="macro", zero_division=0)
    f1_weighted = f1_score(labels, preds, average="weighted", zero_division=0)

    return {
        "accuracy"    : round(acc, 4),
        "f1_macro"    : round(f1_macro, 4),
        "f1_weighted" : round(f1_weighted, 4),
    }

print("Metrics function ready: accuracy, f1_macro, f1_weighted")

---
## 7. Training

The HuggingFace `Trainer` handles:
- **Distributed training** (multi-GPU via `accelerate`)
- **Mixed precision** (FP16 on CUDA)
- **Gradient accumulation** and **gradient clipping**
- **Checkpoint management** and **best-model selection**
- **Evaluation** after each epoch

Training with `EarlyStoppingCallback` prevents over-fitting when the validation F1 stops improving.

In [ ]:
# ── 7.1  Build Trainer ────────────────────────────────────────────────────────
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_tokenized,
    eval_dataset=test_tokenized,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics,
    callbacks=[
        EarlyStoppingCallback(early_stopping_patience=CFG.PATIENCE),
    ],
)

print("Trainer initialised.")
print(f"  Train samples : {len(train_tokenized):,}")
print(f"  Eval  samples : {len(test_tokenized):,}")
print(f"  Total epochs  : {CFG.EPOCHS} (with early stopping, patience={CFG.PATIENCE})")

In [ ]:
# ── 7.2  Fine-tune ────────────────────────────────────────────────────────────
print("Starting fine-tuning...")
print("="*60)

train_result = trainer.train()

print("="*60)
print("Training complete.")
print(f"  Runtime          : {train_result.metrics['train_runtime']:.1f} s")
print(f"  Samples/sec      : {train_result.metrics['train_samples_per_second']:.1f}")
print(f"  Final train loss : {train_result.metrics['train_loss']:.4f}")

# Save the best model and tokenizer
trainer.save_model(CFG.OUTPUT_DIR)
tokenizer.save_pretrained(CFG.OUTPUT_DIR)
print(f"\nBest model saved to: {CFG.OUTPUT_DIR}")

In [ ]:
# ── 7.3  Extract and plot training history ────────────────────────────────────
log_history = trainer.state.log_history

# Separate train and eval logs
train_logs = [l for l in log_history if "loss" in l and "eval_loss" not in l]
eval_logs  = [l for l in log_history if "eval_loss" in l]

train_steps  = [l["step"]  for l in train_logs]
train_losses = [l["loss"]  for l in train_logs]
eval_epochs  = [l["epoch"] for l in eval_logs]
eval_losses  = [l["eval_loss"] for l in eval_logs]
eval_f1s     = [l.get("eval_f1_macro", 0) for l in eval_logs]
eval_accs    = [l.get("eval_accuracy", 0) for l in eval_logs]

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Training loss curve
axes[0].plot(train_steps, train_losses, color="steelblue", linewidth=1.5, alpha=0.8)
axes[0].set_title("Training Loss (per step)", fontsize=13, fontweight="bold")
axes[0].set_xlabel("Step")
axes[0].set_ylabel("Cross-entropy loss")

# Validation loss per epoch
axes[1].plot(eval_epochs, eval_losses, "o-", color="tomato", linewidth=2, markersize=8)
axes[1].set_title("Validation Loss (per epoch)", fontsize=13, fontweight="bold")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Validation loss")
axes[1].set_xticks(eval_epochs)

# Validation metrics per epoch
axes[2].plot(eval_epochs, eval_f1s,  "s-", color="green",  linewidth=2, markersize=8, label="F1 macro")
axes[2].plot(eval_epochs, eval_accs, "^-", color="purple", linewidth=2, markersize=8, label="Accuracy")
axes[2].set_title("Validation Metrics (per epoch)", fontsize=13, fontweight="bold")
axes[2].set_xlabel("Epoch")
axes[2].set_ylabel("Score")
axes[2].set_ylim(0, 1)
axes[2].legend()
axes[2].set_xticks(eval_epochs)

plt.suptitle("DistilBERT Fine-Tuning — Training Curves", fontsize=15, fontweight="bold", y=1.02)
plt.tight_layout()
plt.savefig("../data/training_curves.png", dpi=150, bbox_inches="tight")
plt.show()

---
## 8. Evaluation

Comprehensive evaluation including:
- **Classification report** — precision, recall, F1 per class
- **Confusion matrix** — where does the model confuse classes?
- **ROC curves** — one-vs-rest AUC per class
- **Error analysis** — inspect misclassified examples with attention weights

In [ ]:
# ── 8.1  Get predictions on test set ─────────────────────────────────────────
print("Running predictions on test set...")
predictions = trainer.predict(test_tokenized)

logits      = predictions.predictions          # shape: (N, num_labels)
true_labels = predictions.label_ids            # shape: (N,)
pred_labels = np.argmax(logits, axis=-1)       # shape: (N,)
proba       = torch.softmax(torch.tensor(logits, dtype=torch.float32), dim=-1).numpy()

print(f"Test set   : {len(true_labels):,} examples")
print(f"\nEval metrics: {predictions.metrics}")

In [ ]:
# ── 8.2  Classification report ────────────────────────────────────────────────
target_names = [CFG.ID2LABEL[i] for i in range(CFG.NUM_LABELS)]

report = classification_report(
    true_labels, pred_labels,
    target_names=target_names,
    digits=4,
)
print("Classification Report")
print("=" * 60)
print(report)

# Convert to DataFrame for later use
report_dict = classification_report(
    true_labels, pred_labels,
    target_names=target_names,
    output_dict=True,
)
report_df = pd.DataFrame(report_dict).T.round(4)
print(report_df)

In [ ]:
# ── 8.3  Confusion matrix ─────────────────────────────────────────────────────
cm = confusion_matrix(true_labels, pred_labels)
cm_norm = cm.astype("float") / cm.sum(axis=1, keepdims=True)  # row-normalised

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Raw counts
sns.heatmap(
    cm, annot=True, fmt="d", cmap="Blues",
    xticklabels=target_names, yticklabels=target_names,
    ax=axes[0], linewidths=0.5,
)
axes[0].set_title("Confusion Matrix (counts)", fontsize=13, fontweight="bold")
axes[0].set_xlabel("Predicted")
axes[0].set_ylabel("True")

# Normalised
sns.heatmap(
    cm_norm, annot=True, fmt=".2%", cmap="Blues",
    xticklabels=target_names, yticklabels=target_names,
    ax=axes[1], linewidths=0.5,
)
axes[1].set_title("Confusion Matrix (row-normalised)", fontsize=13, fontweight="bold")
axes[1].set_xlabel("Predicted")
axes[1].set_ylabel("True")

plt.suptitle("DistilBERT — Test Set Confusion Matrices", fontsize=14, fontweight="bold", y=1.02)
plt.tight_layout()
plt.savefig("../data/confusion_matrix.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# ── 8.4  ROC curves (one-vs-rest) ─────────────────────────────────────────────
# Binarise true labels for OvR ROC computation
y_bin = label_binarize(true_labels, classes=[0, 1, 2])

fig, ax = plt.subplots(figsize=(8, 7))
roc_colors = ["crimson", "royalblue", "forestgreen"]

for i, (label, color) in enumerate(zip(target_names, roc_colors)):
    fpr, tpr, _ = roc_curve(y_bin[:, i], proba[:, i])
    roc_auc = auc(fpr, tpr)
    ax.plot(fpr, tpr, color=color, linewidth=2.5,
            label=f"{label} (AUC = {roc_auc:.4f})")

ax.plot([0, 1], [0, 1], "k--", linewidth=1, label="Random classifier")
ax.fill_between([0, 1], [0, 1], alpha=0.05, color="gray")
ax.set_xlim([0.0, 1.0])
ax.set_ylim([0.0, 1.02])
ax.set_xlabel("False Positive Rate", fontsize=12)
ax.set_ylabel("True Positive Rate", fontsize=12)
ax.set_title("ROC Curves — One-vs-Rest (Test Set)", fontsize=14, fontweight="bold")
ax.legend(loc="lower right", fontsize=11)

plt.tight_layout()
plt.savefig("../data/roc_curves.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# ── 8.5  Error analysis — misclassified examples ──────────────────────────────
test_df_reset = test_df.reset_index(drop=True)
test_df_reset["pred_label"]   = pred_labels
test_df_reset["pred_name"]    = test_df_reset["pred_label"].map(CFG.ID2LABEL)
test_df_reset["confidence"]   = proba.max(axis=1)
test_df_reset["correct"]      = test_df_reset["sentiment"] == test_df_reset["pred_label"]

errors = test_df_reset[~test_df_reset["correct"]].sort_values("confidence", ascending=False)
print(f"Misclassified examples: {len(errors)} / {len(test_df_reset)} ({len(errors)/len(test_df_reset):.1%})")
print(f"\nError breakdown by true class:")
print(errors["sentiment_name"].value_counts())

print("\n" + "="*70)
print("TOP 10 HIGH-CONFIDENCE ERRORS (model was very wrong)")
print("="*70)
for _, row in errors.head(10).iterrows():
    print(f"  Text       : {row['text'][:80]}..." if len(row['text']) > 80 else f"  Text       : {row['text']}")
    print(f"  Intent     : {row['intent_name']}")
    print(f"  True label : {row['sentiment_name']}")
    print(f"  Predicted  : {row['pred_name']}  (confidence: {row['confidence']:.2%})")
    print()

In [ ]:
# ── 8.6  Attention weight visualisation on an error example ───────────────────
# DistilBERT outputs attention weights when output_attentions=True

def get_attention_for_text(text: str, model, tokenizer, layer: int = 5) -> tuple:
    """
    Extract mean attention weights from a specific transformer layer.
    Returns (tokens, attention_weights).
    """
    model.eval()
    inputs = tokenizer(
        text, return_tensors="pt",
        truncation=True, max_length=CFG.MAX_LENGTH,
        padding=False,
    ).to(DEVICE)

    with torch.no_grad():
        outputs = model(**inputs, output_attentions=True)

    # outputs.attentions: tuple of (batch, heads, seq, seq) per layer
    # Average over all heads in the selected layer
    attn = outputs.attentions[layer][0]           # (heads, seq, seq)
    attn_mean = attn.mean(dim=0).cpu().numpy()    # (seq, seq)

    # Use attention to [CLS] token (index 0) as token importance
    cls_attn = attn_mean[0, :]                    # how much [CLS] attends to each token

    tokens = tokenizer.convert_ids_to_tokens(inputs["input_ids"][0].cpu())
    return tokens, cls_attn


def plot_attention(tokens, weights, title="Attention Weights"):
    """Bar chart of per-token attention from the [CLS] token."""
    # Remove padding tokens
    valid_mask = [t != "[PAD]" for t in tokens]
    tokens_v   = [t for t, m in zip(tokens, valid_mask) if m]
    weights_v  = np.array([w for w, m in zip(weights, valid_mask) if m])

    # Normalise for colour mapping
    norm_weights = (weights_v - weights_v.min()) / (weights_v.max() - weights_v.min() + 1e-8)

    fig, ax = plt.subplots(figsize=(max(10, len(tokens_v) * 0.55), 3.5))
    bars = ax.bar(range(len(tokens_v)), weights_v,
                  color=plt.cm.Reds(norm_weights), edgecolor="white")
    ax.set_xticks(range(len(tokens_v)))
    ax.set_xticklabels(tokens_v, rotation=45, ha="right", fontsize=9)
    ax.set_ylabel("[CLS] Attention Weight")
    ax.set_title(title, fontsize=12, fontweight="bold")
    plt.tight_layout()
    plt.savefig("../data/attention_example.png", dpi=150, bbox_inches="tight")
    plt.show()


# Pick a high-confidence error for visualisation
if len(errors) > 0:
    error_row = errors.iloc[0]
    error_text = error_row["text"]
    tokens, attn_weights = get_attention_for_text(error_text, model, tokenizer, layer=5)
    title = (
        f"Attention: '{error_text[:50]}...'\n"
        f"True: {error_row['sentiment_name']} | Predicted: {error_row['pred_name']} "
        f"(conf: {error_row['confidence']:.1%})"
    )
    plot_attention(tokens, attn_weights, title=title)
else:
    print("No errors found — perfect test set accuracy!")

In [ ]:
# ── 8.7  Confidence distribution for correct vs. incorrect predictions ─────────
fig, ax = plt.subplots(figsize=(10, 5))

correct_conf   = test_df_reset[test_df_reset["correct"]]["confidence"]
incorrect_conf = test_df_reset[~test_df_reset["correct"]]["confidence"]

ax.hist(correct_conf,   bins=30, alpha=0.7, color="forestgreen", label=f"Correct ({len(correct_conf):,})", density=True)
ax.hist(incorrect_conf, bins=30, alpha=0.7, color="tomato",      label=f"Incorrect ({len(incorrect_conf):,})", density=True)
ax.axvline(0.5, color="black", linestyle="--", linewidth=1.5, label="50% threshold")
ax.set_title("Prediction Confidence Distribution", fontsize=13, fontweight="bold")
ax.set_xlabel("Max softmax probability")
ax.set_ylabel("Density")
ax.legend()

plt.tight_layout()
plt.savefig("../data/confidence_distribution.png", dpi=150, bbox_inches="tight")
plt.show()

print(f"Correct   — mean conf: {correct_conf.mean():.3f}   median: {correct_conf.median():.3f}")
print(f"Incorrect — mean conf: {incorrect_conf.mean():.3f}   median: {incorrect_conf.median():.3f}")

---
## 9. Inference Pipeline

A production-ready inference interface using HuggingFace `pipeline`. This wraps tokenization, model forward pass, and softmax decoding into a single callable.

Key features demonstrated:
- **Single prediction** with confidence scores
- **Batch prediction** for throughput
- **Confidence visualization** for interpretability
- **Uncertainty detection** — flag low-confidence predictions for human review (Argilla active learning loop)

In [ ]:
# ── 9.1  Build inference pipeline ─────────────────────────────────────────────
# Load from saved checkpoint to demonstrate standalone usage
sentiment_pipeline = pipeline(
    task="text-classification",
    model=CFG.OUTPUT_DIR,
    tokenizer=CFG.OUTPUT_DIR,
    device=0 if torch.cuda.is_available() else -1,
    return_all_scores=True,    # return probabilities for all classes
    truncation=True,
    max_length=CFG.MAX_LENGTH,
)

print("Inference pipeline ready.")
print(f"Running on: {'GPU (cuda:0)' if torch.cuda.is_available() else 'CPU'}")

In [ ]:
# ── 9.2  Demo predictions ─────────────────────────────────────────────────────
demo_texts = [
    "My card was declined at the supermarket twice today and I am very frustrated.",
    "Can you tell me what exchange rates you currently offer for euros?",
    "I just activated my new card and the contactless payment worked perfectly!",
    "Someone made an unauthorised transaction on my account last night.",
    "I need to update my billing address on the account.",
    "The cash I withdrew from the ATM was the wrong amount.",
    "How do I link my account to Apple Pay?",
    "My refund was processed and I am happy with the outcome.",
]

print("Sentiment predictions on demo texts:")
print("=" * 75)

results = []
for text in demo_texts:
    output = sentiment_pipeline(text)[0]  # list of {label, score}
    scores = {item["label"]: item["score"] for item in output}
    best_label = max(scores, key=scores.get)
    confidence = scores[best_label]
    results.append({"text": text, "label": best_label, "confidence": confidence, **scores})

    print(f"[{best_label.upper():8s}] ({confidence:.1%}) {text[:65]}")

results_df = pd.DataFrame(results)
print("\nFull score table:")
print(results_df[["label", "confidence", "negative", "neutral", "positive", "text"]].to_string(index=False))

In [ ]:
# ── 9.3  Confidence visualisation ─────────────────────────────────────────────
fig, axes = plt.subplots(2, 4, figsize=(18, 9))
axes = axes.flatten()

sentiment_palette = {"negative": "#E74C3C", "neutral": "#3498DB", "positive": "#2ECC71"}

for i, (ax, row) in enumerate(zip(axes, results_df.itertuples())):
    scores = [row.negative, row.neutral, row.positive]
    bar_colors = [sentiment_palette[l] for l in ["negative", "neutral", "positive"]]
    bars = ax.bar(["neg", "neu", "pos"], scores, color=bar_colors, edgecolor="white", linewidth=1.5)

    # Highlight the predicted class
    pred_idx = ["negative", "neutral", "positive"].index(row.label)
    bars[pred_idx].set_edgecolor("black")
    bars[pred_idx].set_linewidth(3)

    ax.set_ylim(0, 1.05)
    ax.set_ylabel("Probability")
    ax.set_title(
        f"{row.text[:40]}..." if len(row.text) > 40 else row.text,
        fontsize=8, wrap=True,
    )

    for bar, score in zip(bars, scores):
        ax.text(bar.get_x() + bar.get_width()/2, score + 0.02,
                f"{score:.2f}", ha="center", fontsize=8)

plt.suptitle("Sentiment Prediction Confidence per Demo Text",
             fontsize=14, fontweight="bold", y=1.02)
plt.tight_layout()
plt.savefig("../data/inference_confidence.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# ── 9.4  Batch prediction function ────────────────────────────────────────────
def predict_batch(
    texts: list[str],
    pipeline_obj,
    batch_size: int = 32,
    uncertainty_threshold: float = 0.6,
) -> pd.DataFrame:
    """
    Run sentiment prediction on a list of texts in batches.

    Parameters
    ----------
    texts                : List of raw text strings.
    pipeline_obj         : HuggingFace pipeline object.
    batch_size           : Inference batch size.
    uncertainty_threshold: Predictions below this confidence are flagged
                           for human review in Argilla.

    Returns
    -------
    DataFrame with columns: text, label, confidence, negative, neutral,
                             positive, needs_review.
    """
    all_results = []

    for start in range(0, len(texts), batch_size):
        batch    = texts[start : start + batch_size]
        outputs  = pipeline_obj(batch)

        for text, output in zip(batch, outputs):
            scores = {item["label"]: item["score"] for item in output}
            best   = max(scores, key=scores.get)
            conf   = scores[best]
            all_results.append({
                "text"          : text,
                "label"         : best,
                "confidence"    : conf,
                "negative"      : scores.get("negative", 0.0),
                "neutral"       : scores.get("neutral",  0.0),
                "positive"      : scores.get("positive", 0.0),
                "needs_review"  : conf < uncertainty_threshold,
            })

    return pd.DataFrame(all_results)


# Demo: run on 20 test examples
sample_texts = test_df["text"].sample(20, random_state=SEED).tolist()
batch_results = predict_batch(sample_texts, sentiment_pipeline, batch_size=8)

print("Batch prediction results (20 examples):")
print(batch_results[["label", "confidence", "needs_review", "text"]].to_string(index=False))
print(f"\nFlagged for review: {batch_results['needs_review'].sum()} / {len(batch_results)}")

In [ ]:
# ── 9.5  Log uncertain predictions back to Argilla (active learning loop) ─────
uncertain = batch_results[batch_results["needs_review"]]

if ARGILLA_CONNECTED and len(uncertain) > 0:
    uncertain_records = [
        rg.TextClassificationRecord(
            text=row["text"],
            prediction=[
                ("negative", row["negative"]),
                ("neutral",  row["neutral"]),
                ("positive", row["positive"]),
            ],
            prediction_agent=f"{CFG.MODEL_NAME}-finetuned",
            metadata={"confidence": row["confidence"], "iteration": "round-2"},
            id=f"uncertain-{i}",
        )
        for i, (_, row) in enumerate(uncertain.iterrows())
    ]

    rg.log(
        records=uncertain_records,
        name=f"{CFG.ARGILLA_DATASET}-uncertain",
        workspace=CFG.ARGILLA_WORKSPACE,
    )
    print(f"Logged {len(uncertain_records)} uncertain predictions to Argilla for human review.")
    print("Human annotators can now verify these in the Argilla UI and trigger re-training.")
else:
    print(f"[OFFLINE] Would log {len(uncertain)} uncertain examples to Argilla.")
    print("These samples are ideal candidates for the next active learning iteration.")
    if len(uncertain) > 0:
        print("\nUncertain examples:")
        print(uncertain[["text", "label", "confidence"]].to_string(index=False))

In [ ]:
# ── 9.6  Final summary ────────────────────────────────────────────────────────
final_eval = trainer.evaluate(test_tokenized)

print("\n" + "=" * 60)
print("  FINAL MODEL PERFORMANCE SUMMARY")
print("=" * 60)
print(f"  Model         : {CFG.MODEL_NAME} (fine-tuned)")
print(f"  Dataset       : Banking77 → 3-class sentiment")
print(f"  Test accuracy : {final_eval.get('eval_accuracy', 0):.4f}")
print(f"  F1 macro      : {final_eval.get('eval_f1_macro', 0):.4f}")
print(f"  F1 weighted   : {final_eval.get('eval_f1_weighted', 0):.4f}")
print(f"  Test loss     : {final_eval.get('eval_loss', 0):.4f}")
print("=" * 60)

print("\nPer-class metrics:")
for label in target_names:
    p = report_dict[label]
    print(f"  {label:10s}: P={p['precision']:.3f}  R={p['recall']:.3f}  F1={p['f1-score']:.3f}  N={int(p['support'])}")

print("\nSaved artefacts:")
for f in Path("../data").glob("*.png"):
    print(f"  {f}")
print(f"  {CFG.OUTPUT_DIR}/ (model checkpoint)")

---
## Summary & Next Steps

### What we built

| Stage | Component | Key detail |
|-------|-----------|------------|
| Data | Banking77 | 13K queries, 77 intents → 3 sentiment buckets |
| Annotation | Argilla | Human-in-the-loop label verification |
| Model | DistilBERT | 66M params, fine-tuned via Trainer API |
| Training | HuggingFace Trainer | Linear LR warmup, early stopping, FP16 |
| Evaluation | Full suite | Classification report, confusion matrix, ROC, attention |
| Inference | Pipeline | Batch prediction + uncertainty flagging |

### Active learning loop recap
1. **Pretrained DistilBERT** provides baseline predictions on unlabelled data
2. **Argilla** surfaces low-confidence or potentially mislabelled examples for annotators
3. Human-verified labels are pulled back and merged into the training set
4. **Fine-tuned model** learns from the cleaner, human-validated dataset
5. The improved model produces fewer uncertain predictions → fewer annotations needed in the next round

### Possible extensions
- **Multi-label classification** — a query can express multiple intents simultaneously
- **Span-level annotation** in Argilla — highlight the exact phrase driving sentiment
- **LoRA / QLoRA** — parameter-efficient fine-tuning for faster iteration cycles
- **ONNX export** — deploy the model at lower latency without GPU
- **Drift detection** — monitor production confidence distributions and trigger re-labelling when they shift